# Fine-tuning with Unsloth

based on the tutorial:
https://docs.unsloth.ai/basics/tutorial-how-to-finetune-llama-3-and-use-in-ollama



##### In this notebook, we fine-tune models that achieved the fastest inference time in previous benchmarking:

**mistral-7b-v0.3-bnb-4bit**

**Meta-Llama-3.1-8B-bnb-4bit**

**Llama-3.2-3B-bnb-4bit**


In [1]:
import os
import sys
import json
import time
import pandas as pd
from datasets import Dataset
sys.path.insert(1, '../')
from trl import SFTTrainer, SFTConfig
# from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from common import get_test_dataset, get_prompt_template, is_command_classification_correct, to_output_version_1
from unsloth import FastLanguageModel
from unsloth import apply_chat_template
from unsloth import standardize_sharegpt
# import torch
import itertools, json, shutil, os
import wandb
max_seq_length = 2048
dtype = None
load_in_4bit = True # 4bit quantization to reduce memory usage

os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

/home/danielhenel/jarvis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-05 01:29:32.771072: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-05 01:29:32.777519: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746401372.786718 1554803 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746401372.789489 1554803 cuda_blas.cc:1407]

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Prepare train and test subsets

Dataset: Basic commands v2.0

Commands: START, STOP, START_ENGINE, STOP_ENGINE, CRANK_REQUEST

Train 80 - Test 20 split

In [2]:
def get_train_data(tokenizer):
    with open("../data/basic_commands_train_v2.0.json", 'r') as file:
        raw_train_dataset = json.load(file)

    chat_template = """{SYSTEM}
    USER: {INPUT}
    ASSISTANT: {OUTPUT}"""

    system_prompt = get_prompt_template(command_list=["STOP", "START", "ENGINE_STOP", "ENGINE_START", "CRANK_REQUEST"], prompt_version=2)
    system_prompt = system_prompt.replace("'null'", "null")

    train_data = []
    for command_class in raw_train_dataset:
        for input, output in raw_train_dataset[command_class].items():
            if input and output:
                output= output.replace("None", "null")
                output = json.loads(output)
                output = json.dumps(output)
                train_data.append({
                    "conversations": [
                        {"from": "user", "content": str(input)},
                        {"from": "assistant", "content": str(output)}
                    ]
                })

    train_data = Dataset.from_list(train_data)

    train_data = standardize_sharegpt(train_data)

    train_data = apply_chat_template(
        train_data,
        tokenizer = tokenizer,
        chat_template = chat_template,
        default_system_message = system_prompt
    )

    return train_data


def get_test_data():
    with open("../data/basic_commands_test_v2.0.json", 'r') as file:
        raw_test_dataset = json.load(file)
    test_data = []
    for command_class in raw_test_dataset:
        for input, output in raw_test_dataset[command_class].items():
            if input and output:
                output= output.replace("None", "null")
                output = json.loads(output)
                output = json.dumps(output)
                test_data.append({"input": str(input), "command_class": command_class})
    return test_data


### Model, tokenizer and trainer configuration

##### We test various configurations of the following params:
rank(r)

learning rate (lr)

gradient_accumulation_steps (grad_steps) 

number of epochs (epoch)

lora_alpha (alpha)

##### Then we evaluate the accuracy of each model agains our test dataset
##### Finally, we select the model of best accuracy



In [3]:
def get_model_and_tokenizer(model_name: str, r: int, alpha: int) -> tuple:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name =  model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r = r,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj",],
        lora_alpha = alpha,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    return (model, tokenizer)

def get_training_args(lr: float, epoch: int, grad_steps: int, output_dir: str, run_name: str) -> SFTConfig:
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = grad_steps,
        num_train_epochs = epoch,
        learning_rate = lr,
        output_dir=output_dir,
        logging_steps = 5,
        save_strategy="epoch",
        save_total_limit=1,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        report_to = "wandb",
        dataset_text_field = "text",
        dataset_num_proc = 2,
        packing = False,
        max_seq_length=2048,
        run_name=run_name,
        warmup_steps = 5,
        seed = 3407,
    )
    return args

### Training and evaluation

In [4]:
results = {}

def prompt_model(model, tokenizer, messages):
    FastLanguageModel.for_inference(model)

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    output_ids = model.generate(
    input_ids,
    max_new_tokens=128,
    pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = output_ids[0][input_ids.shape[-1]:]
    output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return generated_text

def benchmark_model(model, tokenizer):
    test_dataset = get_test_data()
    predictions = {
        'Classification Result': [],
        'Prompt processing time': []
    }

    probability_threshold = 0.80

    
    for i in range(len(test_dataset)):

        messages=[
            {"role": "system",  "content": get_prompt_template(command_list=["STOP", "START", "ENGINE_STOP", "ENGINE_START", "CRANK_REQUEST"], prompt_version=2)},
            {"role": "user",  "content": test_dataset[i]['input']},]
        
        start_time = time.time() 
        response = prompt_model(model, tokenizer, messages)
        end_time = time.time()
        prompt_processing_time = end_time - start_time 
        answer = to_output_version_1(response)
        classification_result = is_command_classification_correct(answer, [test_dataset[i]["command_class"], None], probability_threshold)
        predictions['Classification Result'].append(classification_result[0])
        predictions['Prompt processing time'].append(prompt_processing_time)

    predictions = pd.DataFrame(predictions)
    accuracy = predictions[predictions['Classification Result'] == "CLASSIFICATION_CORRECT"].shape[0] / predictions.shape[0] 
    accuracy *= 100 # as percentage
    avg_processing_time = predictions['Prompt processing time'].mean()

    return (accuracy, avg_processing_time)

In [5]:
unsloth_4bit_models = [
    "./models/Llama-3.2-3B-bnb-4bit",
    "./models/mistral-7b-v0.3-bnb-4bit",
    "./models/Meta-Llama-3.1-8B-bnb-4bit"
]
learning_rates = [1e-5, 5e-5]
epochs = [2, 3]
grad_acc_steps = [4, 8]
r_values = [16, 32]

param_grid = list(itertools.product(learning_rates, epochs, grad_acc_steps, r_values))

results = []
best_accuracy = -float("inf")
best_model_dir = None

for model_name in unsloth_4bit_models:
    for lr, epoch, grad_steps, r in param_grid:
        alpha = r
        run_name = f"{model_name}_lr{lr}_ep{epoch}_gs{grad_steps}_r{r}_alpha{alpha}"
        output_dir = f"outputs/{run_name}"
        print(f"\nRunning config for {model_name}: {run_name}")

        # Initialize WandB
        wandb.init(mode="offline", name=run_name, reinit=True)

        # Get model and tokenizer
        model, tokenizer = get_model_and_tokenizer(model_name, r, alpha)

        # Get train, test and validation data
        train_dataset = get_train_data(tokenizer) 

        # Train and evaluate
        trainer = SFTTrainer(
            model = model,
            tokenizer = tokenizer,
            train_dataset=train_dataset,
            args = get_training_args(lr, epoch, grad_steps, output_dir, run_name)
        )
        trainer.train()

        # Evaluate on the test dataset
        test_accuracy, avg_processing_time = benchmark_model(model, tokenizer)

        # Log metrics to WandB
        wandb.log({
            "eval_accuracy": test_accuracy,
            "learning_rate": lr,
            "epochs": epoch,
            "grad_acc_steps": grad_steps,
            "r": r,
            "lora_alpha": alpha,
            "avg_processing_time" : avg_processing_time
        })

        # Save to results
        results.append({
            "model_name": model_name,
            "lr": lr,
            "epochs": epoch,
            "grad_acc_steps": grad_steps,
            "r": r,
            "alpha": alpha,
            'eval_accuracy': test_accuracy,
            "output_dir": output_dir,
            "avg_processing_time" : avg_processing_time,
        })

        # Save best model checkpoint
        if test_accuracy > best_accuracy:
            print(f"New best model found at {output_dir} (eval_accuracy={test_accuracy:.4f})")
            best_model_dir = output_dir
            best_accuracy = test_accuracy
            if os.path.exists("best_model_checkpoint"):
                shutil.rmtree("best_model_checkpoint")
            shutil.copytree(output_dir, "best_model_checkpoint")

        wandb.finish()


Running config for ./models/Llama-3.2-3B-bnb-4bit: ./models/Llama-3.2-3B-bnb-4bit_lr1e-05_ep2_gs4_r16_alpha16


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


==((====))==  Unsloth 2025.4.4: Fast Llama patching. Transformers: 4.51.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.505 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Standardizing formats (num_proc=32): 100%|██████████| 682/682 [00:00<00:00, 3549.06 examples/s]
Unsloth: We automatically added an EOS token to stop endless generations.
Map: 100%|██████████| 682/682 [00:00<00:00, 33007.34 examples/s]
/tmp/ipykernel_1554803/1184755956.py:34: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
Converting train dataset to ChatML (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 5154.64 examples/s]
Applying chat template to train dataset (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 1055.43 examples/s]
Truncating train dataset (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 3748.95 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 682 | Num Epochs = 2 | Total steps = 170
O^O/ 

Unsloth: Will smartly offload gradients to save VRAM!


...

==((====))==  Unsloth 2025.4.4: Fast Llama patching. Transformers: 4.51.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.505 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Standardizing formats (num_proc=32): 100%|██████████| 682/682 [00:00<00:00, 2704.41 examples/s]
Unsloth: We automatically added an EOS token to stop endless generations.
Map: 100%|██████████| 682/682 [00:00<00:00, 36712.81 examples/s]
/tmp/ipykernel_1554803/1184755956.py:34: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
Converting train dataset to ChatML (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 3374.94 examples/s]
Applying chat template to train dataset (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 879.72 examples/s]
Truncating train dataset (num_proc=2): 100%|██████████| 682/682 [00:00<00:00, 2699.25 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 682 | Num Epochs = 3 | Total steps = 126
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data

Step,Training Loss
5,1.910600
10,1.562500
15,0.889400
20,0.388000
25,0.191500
30,0.165700
35,0.159100
40,0.162800
45,0.175300
50,0.145400


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


avg_processing_time,▁
epochs,▁
eval_accuracy,▁
grad_acc_steps,▁
learning_rate,▁
lora_alpha,▁
r,▁
train/epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇███
train/global_step,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▇▇▇████
train/grad_norm,▃▃█▂▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁
train/learning_rate,▇██▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▁▁


### Results

In [6]:
results.sort(key=lambda x: x['eval_accuracy'], reverse=True)

for result in results:
    print("Result:")
    for key, value in result.items():
        print(f"  {key}: {value}")
    print("-" * 40)

Result:
  model_name: ./models/mistral-7b-v0.3-bnb-4bit
  lr: 5e-05
  epochs: 2
  grad_acc_steps: 4
  r: 32
  alpha: 32
  eval_accuracy: 76.74418604651163
  output_dir: outputs/./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep2_gs4_r32_alpha32
  avg_processing_time: 0.9205843002297157
----------------------------------------
Result:
  model_name: ./models/mistral-7b-v0.3-bnb-4bit
  lr: 5e-05
  epochs: 3
  grad_acc_steps: 4
  r: 16
  alpha: 16
  eval_accuracy: 76.16279069767442
  output_dir: outputs/./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep3_gs4_r16_alpha16
  avg_processing_time: 0.9171388648277106
----------------------------------------
Result:
  model_name: ./models/mistral-7b-v0.3-bnb-4bit
  lr: 5e-05
  epochs: 2
  grad_acc_steps: 4
  r: 16
  alpha: 16
  eval_accuracy: 75.5813953488372
  output_dir: outputs/./models/mistral-7b-v0.3-bnb-4bit_lr5e-05_ep2_gs4_r16_alpha16
  avg_processing_time: 0.8819990740265957
----------------------------------------
Result:
  model_name: ./models/Meta